# Create Indian Council of Medical Research (ICMR) Awards

Creates OpenAlex award rows from ICMR's published "List of Approved Projects".

**Data source:** ICMR has **no JSON/REST API and publishes no Excel/CSV grant lists.** The structured grant data lives in the monthly "List of International Collaborative Research Projects Approved by HMSC" PDFs linked from `https://www.icmr.gov.in/list-of-approved-projects`. The scraper (`scripts/local/icmr_to_s3.py`) parses the per-month PDFs (2020–2026) plus the consolidated "August 2017 – December 2019" PDF.
**S3 location:** `s3a://openalex-ingest/awards/icmr/icmr_projects.parquet`
**OpenAlex funder:** `F4320320720` — Indian Council of Medical Research, country `IN`, **ROR: none**, DOI `10.13039/501100001411`. F4320* (Path A — present in `openalex.common.funder`).
**Provenance:** `icmr_approved_projects`.
**Priority:** 206.

**⚠️ What ICMR is in this dataset (read before reviewing):** these PDFs are the lists of projects **approved by the Health Ministry's Screening Committee (HMSC)**, which ICMR administers. ICMR/HMSC is the **approving body**, not (usually) the monetary funder — the "Funding/Collaborating Agency" is frequently a foreign/third-party agency (NIH, DBT, a foreign university) or "Self". Under OpenAlex's broad funder model (anything that would appear in a paper's acknowledgements, even with no money provided), HMSC/ICMR approval is acknowledgement-worthy, so these rows map to ICMR. The raw budget shown belongs to the collaborating agency, **not ICMR**, so we deliberately do NOT ship it as `amount`.

**Schema choices:**
- One row per HMSC-approved project. Stable `funder_award_id = icmr-hmsc-{sha1(title|PI|approval_date|agency)}` from the scraper (the source has no native grant ID).
- **`amount` / `currency` are NULL and the Step 6.7 amount check is WAIVED** — the source does not publish an ICMR funding amount (the "Total budget" belongs to the collaborating agency). Do not fabricate amounts. The raw budget + collaborating-agency strings are preserved in the raw table for provenance only.
- `lead_investigator` from the scraper's Python-side name split (`lead_given_name` / `lead_family_name`, post-honorific/suffix strip per runbook §2.4.1 — no SQL splitting here) with the Indian host `institution` as affiliation (country `IN`).
- `funding_type = 'research'`; `funder_scheme = subject_area` (the HMSC "Subject area" label).
- ICMR publishes the approval date (→ `start_year`) but not full start/end dates, so `start_date`/`end_date` remain NULL.
- The large consolidated multi-year volumes (Vol_I 2000–2007 … Vol_IV 2015–2017) are deferred as a follow-up PDF tail (4–5 MB scans on a slow host that stall pdfplumber).

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.icmr_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/icmr/icmr_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_icmr_projects FROM openalex.awards.icmr_raw;

## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `display_name`, `pi_name_raw`, `lead_given_name`, `lead_family_name`, `pi_designation`, `institution`, `collaborating_agency`, `date_of_approval_raw`, `start_year`, `total_budget_raw`, `duration_raw`, `subject_area`, `meeting_label`, `meeting_date_raw`, `source_pdf_url`, `provenance`, `funder_id`, `downloaded_at`.

**Note on amount:** there is intentionally no ICMR amount column. `total_budget_raw` is the collaborating agency's budget (kept for provenance) — it is NOT ICMR's funding and is NOT shipped as `amount`.

In [ ]:
%sql
DESCRIBE openalex.awards.icmr_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, pi_name_raw, lead_given_name, lead_family_name,
       institution, collaborating_agency, date_of_approval_raw, start_year, subject_area,
       total_budget_raw, source_pdf_url
FROM openalex.awards.icmr_raw
LIMIT 10;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(pi_name_raw) AS has_pi,
    COUNT(lead_family_name) AS has_family_name,
    COUNT(institution) AS has_institution,
    COUNT(collaborating_agency) AS has_agency,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(subject_area) AS has_subject_area,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(total_budget_raw) AS has_total_budget_raw_provenance_only
FROM openalex.awards.icmr_raw;

## Step 1.6: Fail-Fast — Verify ICMR Funder Exists (Path A, F4320*)

`F4320320720` is a Crossref-registered (F4320*) funder, so `openalex.common.funder` **must return exactly 1 row**. The transform CROSS JOINs to a `funder_resolved` CTE that reads the dim with a hardcoded fallback (the Abel/MinCiencias canonical-funder gap pattern) so the awards never silently drop to zero.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320320720;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.icmr_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320320720 AS funder_id,
        COALESCE(MAX(display_name), 'Indian Council of Medical Research') AS display_name,
        MAX(ror_id) AS ror_id,                          -- ICMR has no ROR; dim value or NULL
        COALESCE(MAX(doi), '10.13039/501100001411') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320720
),
raw_prepped AS (
    SELECT
        *,
        -- Name split was done in Python (scraper) per runbook §2.4.1; here we
        -- only TRIM/NULLIF the provided columns. No SQL element_at/split.
        NULLIF(TRIM(lead_given_name), '') AS leader_given_name,
        NULLIF(TRIM(lead_family_name), '') AS leader_family_name,
        NULLIF(TRIM(institution), '') AS leader_affiliation_name,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1900 AND 2100
             THEN TRY_CAST(start_year AS INT) END AS start_year_int
    FROM openalex.awards.icmr_raw
)
SELECT
    abs(xxhash64(CONCAT('4320320720:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name,
    CAST(NULL AS STRING) AS description,
    4320320720 AS funder_id,
    rp.funder_award_id,
    -- §6.7 WAIVED: ICMR/HMSC is the approving body. The "Total budget" in the
    -- source belongs to the collaborating/funding agency, NOT ICMR, so we do
    -- not attribute it to ICMR. amount/currency are intentionally NULL.
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    NULLIF(TRIM(rp.subject_area), '') AS funder_scheme,
    'icmr_approved_projects' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    rp.start_year_int AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN rp.leader_family_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                rp.leader_given_name AS given_name,
                rp.leader_family_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CASE WHEN rp.leader_affiliation_name IS NOT NULL THEN 'IN' ELSE NULL END AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.source_pdf_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320320720:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name IS NOT NULL;

## Step 3: Insert into `openalex_awards_raw` at Priority 206

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'icmr_approved_projects' AND priority = 206;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    206 AS priority
FROM openalex.awards.icmr_awards;

## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_icmr_awards FROM openalex.awards.icmr_awards;

In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.icmr_awards;

In [ ]:
%sql
-- Completeness. amount is intentionally 0% (waived — see header / Step 6.7 cell).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    COUNT(start_year) AS has_start_year,
    COUNT(funder_scheme) AS has_funder_scheme,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_affiliation
FROM openalex.awards.icmr_awards;

In [ ]:
%sql
-- Step 6.7 amount/currency coverage — WAIVED for ICMR.
-- ICMR/HMSC is the approving body; the source publishes no ICMR funding amount
-- (the "Total budget" belongs to the collaborating agency). pct_amount = 0 and
-- distinct_currencies = 0 are EXPECTED here and are NOT a failure. Do not
-- "fix" this by mapping total_budget_raw — that money is not ICMR's.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.icmr_awards;

In [ ]:
%sql
-- PI frequency (Step 6.4a): no single name should dominate; no institution-as-name.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.icmr_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards
FROM openalex.awards.icmr_awards
GROUP BY funding_type ORDER BY awards DESC;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title, start_year,
       lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 55) AS affiliation,
       SUBSTRING(funder_scheme, 1, 30) AS subject_area
FROM openalex.awards.icmr_awards
ORDER BY start_year DESC, id LIMIT 20;

In [ ]:
%sql
-- Confirm the priority insert landed (Step 6.8).
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'icmr_approved_projects'
GROUP BY priority, provenance;